In [1]:
import os
import json
import polars as pl

In [2]:
def load_json_file(file):
    with open(file) as f:
        data = json.load(f)
    return data['data']

In [3]:
load_json_file('cro/plus-2024-03-24.json')

[]

In [4]:
with open('cro/radiowave-2025-03-24.json', 'r', encoding='utf-8') as f:
    print(f.read())


{"timestamp":"2025-03-24T23:59:36+01:00","data":[{"since":"2025-03-24T00:01:53+01:00","id":21856162,"interpret":"LYNKS","interpret_id":33859,"track":"Tennis Song","track_id":114355,"itemcode":"9779240","files":[{"source":"gselector","id":"9779240","asset":"http:\/\/data.rozhlas.cz\/api\/v2\/asset\/cover\/gselector\/9779240.jpg","asset_width":240,"asset_height":240}]},{"since":"2025-03-24T00:04:02+01:00","id":21856168,"interpret":"LA ROUX","interpret_id":8484,"track":"Automatic Driver","track_id":76671,"itemcode":"9777375","files":[{"source":"gselector","id":"9777375","asset":"http:\/\/data.rozhlas.cz\/api\/v2\/asset\/cover\/gselector\/9777375.jpg","asset_width":240,"asset_height":240}]},{"since":"2025-03-24T00:09:04+01:00","id":21856175,"interpret":"MATT BERRY","interpret_id":24111,"track":"Phantom Birds","track_id":83306,"itemcode":"9777895","files":[{"source":"gselector","id":"9777895","asset":"http:\/\/data.rozhlas.cz\/api\/v2\/asset\/cover\/gselector\/9777895.jpg","asset_width":24

In [5]:
df = pl.DataFrame()

for soubor in os.listdir("cro"):
    if soubor.endswith(".json"):
        try:
            dzejsn = load_json_file(os.path.join('cro', soubor))
        except Exception as e:
            print(f"{soubor}: {e}")
        try:
            stanicoden = pl.DataFrame(
                dzejsn, 
                schema=pl.Schema(
                    {
                        'since':pl.String,
                        'id':pl.String,
                        'interpret':pl.String,
                        'track':pl.String,
                        'track_id':pl.String,
                        'stanice':pl.String,
                        'interpret_id':pl.String,
                        'itemcode':pl.String
                    }
                )
            )
        except Exception as e:
            print(f"{soubor}: {e}")
        stanice = soubor.split("-2")[0]
        if len(stanicoden) > 0:
            try:
                df = pl.concat([df, stanicoden.with_columns(pl.lit(stanice).alias('stanice'))])
            except Exception as e:
                print(f"{soubor}: {e}")

df = df.with_columns(pl.col('since').str.to_datetime()).drop('files', strict=False).with_columns(pl.col('stanice').cast(pl.Categorical)).sort(by='since')

dvojka-2021-02-22.json: Expecting value: line 3 column 1 (char 2)
dvojka-2021-02-24.json: Expecting value: line 3 column 1 (char 2)
plus-2021-02-24.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-01.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-02.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-03.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-04.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-05.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-06.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-07.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-08.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-09.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-10.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-11.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-12-12.json: Expecting value: line 3 column 1 (char 2)
pohoda-2020-

In [10]:
df.sample(10)

since,id,interpret,track,track_id,stanice,interpret_id,itemcode
"datetime[μs, UTC]",str,str,str,str,cat,str,str
2024-10-14 03:39:10 UTC,"""21074393""","""GAZEBO""","""I Like Chopin""","""7599""","""dvojka""","""3092""","""2020293"""
2023-05-19 05:20:03 UTC,"""17861044""","""KARI FAUX""","""Look At That""","""84678""","""radiowave""","""19340""","""9777963"""
2024-02-01 14:43:45 UTC,"""19866967""","""CALLING""","""WHEREVER YOU WILL GO""","""10464""","""radiozurnal""","""4227""","""1004706"""
2024-03-23 16:25:58 UTC,"""20113521""","""P!NK""","""WHO KNEW""","""8146""","""radiozurnal""","""3225""","""1006578"""
2024-01-19 02:56:56 UTC,"""19800997""","""JIŘÍ SUCHÝ""","""Léta dozrávání""","""11229""","""pohoda""","""3595""","""8300189"""
2024-02-17 09:20:41 UTC,"""19942364""","""LUBOŠ SOUKUP QUARTET""","""Folk Song""","""14101""","""jazz""","""5114""","""GS04336"""
2022-04-26 03:15:07 UTC,"""13631889""","""MISSY ELLIOT""","""Throw It Back""","""70612""","""radiowave""","""21321""","""9777143"""
2024-10-31 20:56:26 UTC,"""21161911""","""THE BEATLES""","""Yesterday""","""20969""","""dvojka""","""7178""","""2020248"""
2025-01-22 16:22:34 UTC,"""21560966""","""DIAZPORA""","""Yaroslava""","""40663""","""jazz""","""11720""","""GS07527"""


In [6]:
df.write_parquet(os.path.join("data","cro.parquet"), use_pyarrow=True)

In [8]:
df.filter(pl.col('since').dt.hour() == 0)

since,id,interpret,track,track_id,stanice,interpret_id,itemcode
"datetime[μs, UTC]",str,str,str,str,cat,str,str
2020-12-01 00:01:16 UTC,"""10236339""","""POEMSS""","""Gentle Mirror""","""23531""","""radiowave""","""7971""",null
2020-12-01 00:02:34 UTC,"""10236343""","""DES REE""","""LIFE""","""11948""","""dvojka""","""4640""",null
2020-12-01 00:02:34 UTC,"""10236341""","""FRED WESLEY""","""Wuda Cuda Shuda""","""32104""","""jazz""","""10035""",null
2020-12-01 00:04:46 UTC,"""10236346""","""SHURA""","""Just Once""","""26311""","""radiowave""","""8695""",null
2020-12-01 00:05:58 UTC,"""10236349""","""ARGEMA""","""KLIDNÁ HOLKA""","""40036""","""dvojka""","""3965""",null
…,…,…,…,…,…,…,…
2025-06-03 00:55:16 UTC,"""22199624""","""MILES DAVIS""","""Red""","""572""","""jazz""","""299""","""GS00628"""
2025-06-03 00:56:43 UTC,"""22199628""","""CRASH TEST DUMMIES""","""MMM MMM MMM MMM""","""8029""","""dvojka""","""3327""","""9900013"""
2025-06-03 00:56:59 UTC,"""22199630""","""FAITH HILL""","""THIS KISS""","""12682""","""radiozurnal""","""3326""","""1004160"""
